# **Uczenie maszynowe - Lab_2**

---


Student:
*   podejmuje decyzje preprocessingowe na podstawie wcześniejszej inspekcji (U_01),
*   poprawnie dzieli dane i unika data leakage (U_01),
*   buduje pierwszy model klasyfikacyjny (U_02),
*   interpretuje accuracy w kontekście niezbalansowanych klas (U_04),
*   analizuje wpływ skalowania na wynik (U_03).

---



$\color{red}{Uwaga:}$

1- proszę pamiętać o zmianie słowa ***Album*** w nazwie pliku i uzupełnieniu swoich danych powyżej tego pola;

2- kluczowe polecenia proszę opatrzyć komentarzem;

3- wnioski, pod zadaniem, wpisujemy w polu tekstowym.

In [ ]:
# Podpisanie pracy
NN = input("Podaj Imię i Nazwisko: ")
ALBUM = input("Podaj numer albumu: ")

Podaj Imię i Nazwisko: illia semenov
Podaj numer albumu: 50779


In [1]:
# Montowanie dysku Google Drive w środowisku Google Colab
# Pozwala to na dostęp do plików zapisanych na dysku użytkownika

from google.colab import drive

# Uruchomienie procesu montowania
# Po wykonaniu tej komendy pojawi się link autoryzacyjny
drive.mount('/content/drive')

Mounted at /content/drive


Zad 1. Decyzje preprocessingowe. Rekapitulacja Lab 1

1. Które zmienne mają braki?
W zbiorze danych Titanic największa liczba braków występuje w kolumnie age, gdzie część wartości wieku pasażerów nie została zarejestrowana. Dodatkowo kolumna deck zawiera dużą liczbę brakujących wartości, natomiast w kolumnie embarked braki występują sporadycznie. Przed trenowaniem modeli konieczne jest uzupełnienie braków — dla zmiennych numerycznych można zastosować np. medianę, natomiast dla zmiennych kategorycznych tryb lub osobną kategorię oznaczającą brak danych.

2. Które zmienne są kategoryczne?
Do zmiennych kategorycznych w zbiorze Titanic należą m.in. sex, class, who, deck, embarked, alive oraz adult_male. Zmienne te przyjmują wartości dyskretne i nie mogą być bezpośrednio używane w modelach uczenia maszynowego operujących na danych liczbowych. Wymagają one odpowiedniego kodowania, np. one-hot encoding.

3. Czy klasy są zbalansowane?
Zbiór danych jest niezbalansowany względem zmiennej docelowej survived. Klasa „nie przeżył” (0) występuje częściej niż klasa „przeżył” (1). W takich warunkach model może preferować klasę większościową, osiągając wysoką wartość accuracy, mimo że słabo radzi sobie z wykrywaniem klasy mniejszościowej. W celu poprawy jakości modelu można zastosować techniki takie jak oversampling, undersampling lub nadanie wag klasom.

4. Czy zmienne są w podobnych skalach?
Zmienne numeryczne w zbiorze Titanic nie są w tej samej skali. Przykładowo, pclass przyjmuje wartości od 1 do 3, age mieści się w zakresie od 0 do około 80, natomiast fare może osiągać wartości powyżej 500. W modelach opartych na odległości (np. KNN) takie różnice powodują, że zmienne o większym zakresie dominują w obliczeniach odległości euklidesowej. Dlatego konieczne jest zastosowanie standaryzacji lub normalizacji cech, aby zapewnić równy wpływ wszystkich zmiennych na model.

### **Zad 2.** Wybór atrybutów (cech)

In [2]:
import seaborn as sns
import pandas as pd

# Wczytanie wbudowanego zbioru danych Titanic z biblioteki seaborn
df = sns.load_dataset("titanic")

# Wybór cech (features), które będą używane do trenowania modelu
features = ["pclass", "sex", "age", "fare"]

# Zmienna docelowa (target), którą model będzie przewidywał
target = "survived"

# Ograniczenie DataFrame do wybranych kolumn (cechy + target)
df = df[features + [target]]

# Wyświetlenie pierwszych wierszy dla weryfikacji danych
df.head()

,pclass,sex,age,fare,survived
0,3,male,22.0,7.2500,0
1,1,female,38.0,71.2833,1
2,3,female,26.0,7.9250,1
3,1,female,35.0,53.1000,1
4,3,male,35.0,8.0500,0


In [3]:
df

,pclass,sex,age,fare,survived
0,3,male,22.0,7.2500,0
1,1,female,38.0,71.2833,1
2,3,female,26.0,7.9250,1
3,1,female,35.0,53.1000,1
4,3,male,35.0,8.0500,0
...,...,...,...,...,...
886,2,male,27.0,13.0000,0
887,1,female,19.0,30.0000,1
888,3,female,NaN,23.4500,0
889,1,male,26.0,30.0000,1


Dyskusja wyników
1. Dlaczego nie bierzemy wszystkich zmiennych?

Nie wszystkie zmienne dostępne w zbiorze danych są użyteczne dla modelu predykcyjnego. Część z nich:

zawiera dużą liczbę brakujących wartości (np. deck),
jest redundantna lub powiela informacje (np. alive względem survived),
nie wnosi istotnej wartości predykcyjnej (np. name, ticket w zbiorze Titanic),
może wprowadzać szum i utrudniać proces uczenia modelu.

Uwzględnienie wszystkich zmiennych bez wcześniejszej selekcji może prowadzić do:

nadmiernego dopasowania modelu (overfitting),
wydłużenia czasu trenowania,
obniżenia jakości i interpretowalności wyników.
2. Czym kierujemy się przy wyborze atrybutów?

Proces wyboru cech (feature selection) opiera się na kilku kryteriach:

Braki danych – cechy z dużym udziałem braków mogą wymagać imputacji lub zostać odrzucone, jeśli ich informacyjność jest niewielka.
Typ zmiennych – zmienne kategoryczne i numeryczne wymagają odpowiedniego przygotowania (np. kodowania).
Korelacja i redundancja – silnie skorelowane zmienne niosą podobną informację, dlatego często jedną z nich można usunąć.
Informacyjność – cechy o niskiej zmienności lub mało zróżnicowane dostarczają ograniczonej informacji dla modelu.
Znaczenie merytoryczne – wiedza dziedzinowa pozwala ocenić, które zmienne mogą mieć realny wpływ na zmienną docelową.

### **Zad 3.** Podział danych

In [5]:
from sklearn.model_selection import train_test_split

# Przygotowanie danych wejściowych oraz zmiennej docelowej
# X zawiera wszystkie cechy opisujące obserwacje,
# natomiast y reprezentuje zmienną, którą chcemy przewidywać

features = df.drop(columns=[target])
labels = df[target]

# Podział zbioru danych na część treningową i testową
# Utrzymujemy proporcje klas dzięki parametrowi stratify
X_train, X_test, y_train, y_test = train_test_split(
    features,
    labels,
    test_size=0.2,
    random_state=10,
    stratify=labels
)
# Dane zostały podzielone na zbiór treningowy oraz testowy w proporcji 80/20.

# Zastosowanie parametru stratify pozwala zachować podobny rozkład klas
# w obu zbiorach, co jest istotne w przypadku problemów klasyfikacyjnych.

# Dzięki temu model będzie trenowany na reprezentatywnych danych,
# a jego ocena na zbiorze testowym będzie bardziej wiarygodna.

In [10]:
train_size = X_train.shape
test_size = X_test.shape

print(f"Zbiór treningowy zawiera {train_size[0]} obserwacji i {train_size[1]} cech.")
print(f"Zbiór testowy zawiera {test_size[0]} obserwacji i {test_size[1]} cech.")

Zbiór treningowy zawiera 712 obserwacji i 4 cech.
Zbiór testowy zawiera 179 obserwacji i 4 cech.


Dyskusja wyników
1. Dlaczego najpierw podział, a potem przetwarzanie?

Podział danych na zbiór treningowy i testowy powinien być wykonany przed etapem przetwarzania, aby uniknąć zjawiska tzw. data leakage (przecieku informacji).

Gdyby operacje takie jak imputacja braków danych (np. uzupełnianie medianą wieku) zostały wykonane na całym zbiorze przed podziałem, to informacje z danych testowych wpływałyby na dane treningowe. W konsekwencji model byłby oceniany na danych, które w pewnym stopniu „uczestniczyły” w procesie przygotowania danych, co mogłoby sztucznie zawyżyć wyniki i prowadzić do błędnych wniosków o jego skuteczności.

2. Dlaczego stosujemy parametr stratify?

Parametr stratify=y w funkcji train_test_split zapewnia zachowanie proporcji klas w zbiorach treningowym i testowym zgodnie z ich rozkładem w całym zbiorze danych.

Jest to szczególnie istotne w przypadku danych niezbalansowanych (np. zmienna survived), ponieważ:

model uczy się na reprezentatywnym rozkładzie klas,
zbiór testowy wiernie odzwierciedla rzeczywiste proporcje,
wyniki ewaluacji są bardziej wiarygodne.

Bez stratifikacji mogłoby dojść do sytuacji, w której jedna z klas (np. mniejszościowa) byłaby niedostatecznie reprezentowana lub nawet pominięta w jednym ze zbiorów.

3. Dlaczego nie wypełniamy braków przed podziałem?

Braki danych powinny być uzupełniane po podziale na zbiory, przy czym parametry uzupełniania (np. średnia, mediana) powinny być wyznaczane wyłącznie na podstawie zbioru treningowego.

Wypełnianie braków przed podziałem prowadzi do przecieku informacji, ponieważ statystyki obliczone na całym zbiorze zawierają również informacje z danych testowych. W efekcie model otrzymuje pośrednio wiedzę o danych, na których powinien być później oceniany, co zaburza rzetelność wyników.

### **Zad 4.** Imputacja braków

In [11]:
# Wyznaczenie wartości środkowej (mediany) wieku na podstawie danych treningowych
median_age = X_train["age"].median()

# Zastąpienie brakujących wartości w wieku medianą wyliczoną z treningu
X_train.loc[:, "age"] = X_train["age"].fillna(median_age)
X_test.loc[:, "age"] = X_test["age"].fillna(median_age)

Dyskusja wyników:
Używamy medianę obliczoną tylko na zbiorze treningowym, aby uniknąć przecieku informacji (data leakage) z danych testowych.

🔹 Wyjaśnienie:

Zbiór treningowy służy do uczenia modelu i wyznaczania wartości statystycznych (np. mediany wieku).

Dane testowe powinny pozostać nienaruszone i służyć wyłącznie do oceny modelu.

Gdybyśmy użyli mediany z całego zbioru (trening + test), model „widzi” części informacji z testu już na etapie uczenia, co może sztucznie poprawić wyniki i dać nierealistyczną dokładność.
---



### **Zad 5.** Kodowanie zmiennej kategorycznej

In [12]:
# Zamiana zmiennej kategorycznej 'sex' na zmienne numeryczne (one-hot encoding)
X_train = pd.get_dummies(X_train, columns=["sex"], drop_first=True)
X_test = pd.get_dummies(X_test, columns=["sex"], drop_first=True)

# Wyświetlenie podstawowych informacji o strukturze danych treningowych
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 712 entries, 415 to 253
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   pclass    712 non-null    int64  
 1   age       712 non-null    float64
 2   fare      712 non-null    float64
 3   sex_male  712 non-null    bool   
dtypes: bool(1), float64(2), int64(1)
memory usage: 22.9 KB


Dyskusja wyników:
Czy sposób zakodowania zmiennej kategorycznej może wpływać na geometrię przestrzeni cech? Uzasadnij.

Tak, sposób kodowania zmiennych kategorycznych ma bezpośredni wpływ na geometrię przestrzeni cech, a tym samym na działanie modeli opartych na odległości (np. KNN, SVM).

🔹 Wyjaśnienie:

One-hot encoding

Każda kategoria staje się osobną kolumną binarną (0/1).

Wartości są równomiernie rozłożone i odległość euklidesowa między różnymi kategoriami jest stała.

Przykład: sex → male=1, female=0 (po drop_first).

Label encoding

Kategorie są przypisane do liczb całkowitych (np. male=0, female=1).

Wprowadza sztuczny porządek liczbowy – odległość między 0 i 1 traktowana jest jak różnica wartości liczbowych.

W modelach opartych na odległości może to zniekształcić relacje między obserwacjami i dawać błędne wyniki.

🔹 Wniosek:

Kodowanie zmiennych kategorycznych bezpośrednio zmienia kształt i odległości w przestrzeni cech.

Dlatego dla modeli opartych na odległościach zaleca się one-hot encoding, aby zachować równe traktowanie kategorii i poprawną geometrię przestrzeni cech.


### **Zad 6.** Model bazowy

In [13]:
import numpy as np
from sklearn.metrics import accuracy_score

# Określenie najczęściej występującej klasy w zbiorze treningowym
majority_class = y_train.value_counts().idxmax()

# Utworzenie predykcji bazowej – przypisanie wszystkich obserwacji do klasy dominującej
y_pred_baseline = np.full_like(y_test, majority_class)

# Obliczenie dokładności dla modelu bazowego
baseline_accuracy = accuracy_score(y_test, y_pred_baseline)

print(f"Dokładność modelu bazowego: {baseline_accuracy:.4f}")

Dokładność modelu bazowego: 0.6145


Dyskusja wyników
1. Czym jest model bazowy zastosowany w tym ćwiczeniu?

Model bazowy (ang. baseline model) to bardzo prosty punkt odniesienia, który nie wykorzystuje żadnych zależności między zmiennymi wejściowymi. Jego działanie polega na przewidywaniu jednej, najczęściej występującej klasy w zbiorze treningowym dla wszystkich obserwacji.

W analizowanym przypadku dla zbioru Titanic model taki przypisuje każdemu pasażerowi klasę dominującą w zbiorze treningowym y_train, czyli „nie przeżył” (0).

Główną rolą tego podejścia jest dostarczenie wartości referencyjnej, względem której można porównać bardziej złożone modele i ocenić, czy faktycznie wnoszą one poprawę.

2. Interpretacja wyniku w kontekście rozkładu klas w zbiorze Titanic

W zbiorze Titanic obserwuje się nierównomierny rozkład klas — większość pasażerów nie przeżyła (około 60–63%), natomiast mniejsza część przeżyła (około 37–40%).

Model bazowy, który zawsze przewiduje klasę 0, osiąga dokładność na poziomie zbliżonym do udziału tej klasy w danych, czyli około 60–62%.

Oznacza to, że uzyskany wynik nie wynika z faktycznego „uczenia się”, lecz z przewagi liczebnej jednej z klas. Model nie rozróżnia pasażerów pod względem cech — jedynie odtwarza rozkład dominującej klasy.

3. Czy 60% accuracy to dobry wynik?

W tym przypadku dokładność na poziomie około 60% nie powinna być interpretowana jako dobra jakość modelu. Wynika to z faktu, że model nie wykorzystuje żadnych informacji zawartych w danych wejściowych i nie dokonuje realnej klasyfikacji.

W sytuacji niezbalansowanych klas metryka accuracy może być myląca, ponieważ wysoka wartość nie gwarantuje poprawnego rozpoznawania klasy mniejszościowej.

Dlatego bardziej wiarygodną ocenę jakości modelu zapewniają inne metryki, takie jak precision, recall, F1-score czy macierz pomyłek, które pozwalają lepiej ocenić skuteczność modelu względem obu klas.

### **Zad 7.** Pierwszy model (k-NN)

In [14]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# Inicjalizacja modelu KNN z określoną liczbą sąsiadów
knn_model = KNeighborsClassifier(n_neighbors=5)

# Dopasowanie modelu do danych treningowych
knn_model.fit(X_train, y_train)

# Wykonanie predykcji na danych testowych i obliczenie dokładności
y_pred = knn_model.predict(X_test)
model_accuracy = accuracy_score(y_test, y_pred)

print(f"Dokładność modelu KNN: {model_accuracy:.6f}")

Dokładność modelu KNN: 0.731844


### **Zad 8.** Skalowanie

In [15]:
from sklearn.preprocessing import StandardScaler

# Utworzenie obiektu do standaryzacji danych
scaler = StandardScaler()

# Dopasowanie skalera do danych treningowych i transformacja tych danych
X_train_scaled = scaler.fit_transform(X_train)

# Transformacja danych testowych przy użyciu wcześniej dopasowanego skalera
X_test_scaled = scaler.transform(X_test)

Dyskusja wyników
Dlaczego procedura skalowania przebiega w taki sposób?

Proces skalowania danych w uczeniu maszynowym realizowany jest według określonego schematu, którego celem jest zachowanie poprawności eksperymentu oraz uniknięcie przecieku informacji.

Na początku dokonuje się podziału danych na zbiór treningowy i testowy. Jest to kluczowe, ponieważ dane testowe powinny pozostać całkowicie niezależne i służyć wyłącznie do oceny końcowej modelu, bez wpływu na etap jego uczenia.

Następnie skaler (np. StandardScaler) jest dopasowywany wyłącznie do danych treningowych. W tym kroku wyznaczane są parametry takie jak średnia oraz odchylenie standardowe. Dzięki temu informacje pochodzące ze zbioru testowego nie są wykorzystywane w procesie przygotowania modelu.

Kolejnym etapem jest przekształcenie zbioru treningowego przy użyciu wyznaczonych parametrów, tak aby cechy miały ustandaryzowaną skalę (średnia równa 0 i odchylenie standardowe równe 1).

Analogicznie transformacji poddawany jest zbiór testowy, jednak wykorzystuje się do tego parametry obliczone wcześniej na danych treningowych. Nie wykonuje się ponownego dopasowania skalera na danych testowych, ponieważ prowadziłoby to do naruszenia zasad poprawnej walidacji.

Głównym powodem takiego podejścia jest eliminacja zjawiska data leakage, czyli sytuacji, w której informacje z danych testowych w sposób pośredni wpływają na proces uczenia modelu. Dzięki temu model uczy się wyłącznie na danych treningowych, a jego ocena na zbiorze testowym pozostaje wiarygodna i odzwierciedla rzeczywistą zdolność generalizacji.

### **Zad 9.** Drugi model (k-NN)

In [16]:
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# Inicjalizacja obiektu do standaryzacji danych
scaler = StandardScaler()

# Dopasowanie skalera do danych treningowych i ich przekształcenie
X_train_scaled = scaler.fit_transform(X_train)

# Przekształcenie danych testowych przy użyciu parametrów z treningu
X_test_scaled = scaler.transform(X_test)

# Utworzenie i trenowanie modelu KNN
knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train_scaled, y_train)

# Predykcja oraz ocena dokładności modelu
y_pred = knn_model.predict(X_test_scaled)
model_accuracy = accuracy_score(y_test, y_pred)

print(f"Dokładność modelu KNN po skalowaniu: {model_accuracy:.12f}")

Dokładność modelu KNN po skalowaniu: 0.804469273743


Dyskusja wyników
1. Czy wynik uległ zmianie?

Tak, zastosowanie skalowania danych zazwyczaj wpływa na poprawę jakości działania modelu KNN.

W przypadku danych przed standaryzacją cechy o większych wartościach liczbowych, takich jak fare, miały znacznie większy wpływ na obliczanie odległości. W efekcie model mógł w dużej mierze ignorować pozostałe zmienne, takie jak pclass czy age.

Po przeprowadzeniu standaryzacji wszystkie cechy zostają przekształcone do wspólnej skali (średnia równa 0 oraz odchylenie standardowe równe 1). Dzięki temu każda zmienna wnosi porównywalny wkład do obliczeń, co umożliwia modelowi bardziej efektywne wykorzystanie pełnej informacji zawartej w danych.

2. Dlaczego zmienna fare może dominować?

W oryginalnym zbiorze wartości zmiennej fare są znacznie większe (od wartości bliskich 0 aż do kilkuset), podczas gdy inne cechy mają znacznie mniejsze zakresy — np. pclass przyjmuje wartości od 1 do 3, a age mieści się w przedziale około 0–80.

Modele takie jak KNN wykorzystują metryki odległości (np. euklidesową), dlatego różnice w skali cech bezpośrednio wpływają na wynik obliczeń. W praktyce oznacza to, że zmienne o większym zakresie wartości, takie jak fare, mogą dominować w wyznaczaniu odległości między obserwacjami.

W konsekwencji pozostałe cechy mają mniejszy wpływ na proces klasyfikacji.

💡 Z tego powodu standaryzacja danych jest szczególnie istotna w modelach opartych na odległościach — zapewnia ona, że wszystkie cechy są traktowane na równych zasadach i żadna z nich nie dominuje ze względu na swoją skalę.

### **Zad 10.** Pipeline

$\color{red}{Uwaga:}$

Pipeline jest operatorem złożonym z transformatorów i estymatora końcowego, umożliwiającym walidację i strojenie całego procesu jako jednej struktury funkcjonalnej.

In [17]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# Definicja pipeline łączącego standaryzację i klasyfikator KNN
knn_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=5))
])

# Trenowanie modelu na danych treningowych
knn_pipeline.fit(X_train, y_train)

# Predykcja i ocena skuteczności modelu
y_pred = knn_pipeline.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Dokładność modelu KNN z pipeline: {accuracy:.6f}")

Dokładność modelu KNN z pipeline: 0.804469


Dyskusja wyników
Czy wynik uległ zmianie?

Tak, zastosowanie standaryzacji danych wpływa na zmianę, a zazwyczaj także na poprawę jakości działania modelu KNN.

W przypadku danych bez wcześniejszego skalowania cechy o dużych wartościach liczbowych, takie jak fare, mają większy wpływ na obliczanie odległości euklidesowej. W konsekwencji model może w ograniczonym stopniu uwzględniać pozostałe zmienne wejściowe.

Po przekształceniu danych do wspólnej skali wszystkie cechy uzyskują średnią równą 0 oraz odchylenie standardowe równe 1. Dzięki temu ich wpływ na obliczanie odległości staje się bardziej wyrównany.

W efekcie model KNN jest w stanie lepiej wykorzystać informacje zawarte we wszystkich cechach, co często przekłada się na wyższą skuteczność predykcji na zbiorze testowym

### Wnioski końcowe


Dlaczego preprocessing powinien być dopasowany wyłącznie do zbioru treningowego?

Wszystkie operacje przygotowania danych, takie jak obliczanie statystyk (np. średniej, mediany czy odchylenia standardowego), powinny być wykonywane wyłącznie na danych treningowych. Pozwala to uniknąć zjawiska tzw. data leakage, czyli niezamierzonego „przenikania” informacji z danych testowych do procesu uczenia modelu.

Jeżeli wykorzystamy dane testowe podczas dopasowywania transformacji (np. imputacji lub skalowania), model uzyska dostęp do informacji, które w rzeczywistych warunkach nie byłyby dostępne. W konsekwencji może to prowadzić do zawyżonych i nierealistycznych wyników oceny.

Czy accuracy jest dobrą metryką przy niezbalansowanych klasach?

Metryka accuracy nie zawsze dobrze odzwierciedla jakość modelu w przypadku danych o nierównym rozkładzie klas. W zbiorze takim jak Titanic, gdzie jedna z klas (np. „nie przeżył”) dominuje liczebnie, model może osiągać wysoką dokładność, przewidując zawsze klasę większościową.

W praktyce oznacza to, że nawet prosty model bazowy może uzyskać wynik około 60%, mimo że nie uczy się żadnych zależności między cechami. Z tego powodu accuracy może być myląca i nie powinna być jedyną miarą oceny.

Lepszy obraz jakości modelu uzyskuje się, analizując dodatkowe metryki, takie jak precision, recall, F1-score czy macierz pomyłek, które pozwalają ocenić skuteczność modelu również względem klasy mniejszościowej.

Jak skalowanie wpływa na modele oparte na odległości?

Algorytmy wykorzystujące miary odległości, takie jak KNN czy SVM z jądrem RBF, są szczególnie wrażliwe na różnice w skali cech. Jeśli jedna z cech przyjmuje znacznie większe wartości (np. fare), może ona dominować w obliczeniach odległości, ograniczając wpływ pozostałych zmiennych.

Standaryzacja danych rozwiązuje ten problem poprzez sprowadzenie wszystkich cech do wspólnej skali. Dzięki temu każda zmienna ma porównywalny wpływ na wynik obliczeń, co sprzyja bardziej zrównoważonemu wykorzystaniu informacji i często prowadzi do lepszych rezultatów modelu.

Czy model bazowy może pełnić rolę punktu odniesienia?

Tak, model bazowy stanowi ważny punkt referencyjny w ocenie modeli uczenia maszynowego. Jego zadaniem nie jest predykcja oparta na danych wejściowych, lecz jedynie przewidywanie najczęściej występującej klasy.

Jeżeli bardziej zaawansowany model nie osiąga wyników lepszych niż baseline, oznacza to, że nie wnosi on dodatkowej wartości lub wymaga dalszej optymalizacji.

Co stanowiło największe ryzyko błędu w tej procedurze?

Największym zagrożeniem była możliwość wystąpienia data leakage, czyli wykorzystania informacji z danych testowych na etapie przygotowania danych lub trenowania modelu.

Drugim istotnym źródłem potencjalnych problemów była niewłaściwa transformacja zmiennych kategorycznych. Zastosowanie nieodpowiedniego kodowania mogłoby zaburzyć relacje między obserwacjami i negatywnie wpłynąć na działanie modeli opartych na odległości.